In [ ]:
"""
PMC XML Data Extraction Pipeline
==================================
Merged from: marlo-80_parser_2.py + 01_carpetcrawler78-data_extraction.ipynb

Output: raw_articles.pkl
Cleaning and EDA: see notebooks/02_cleaning_eda.ipynb
"""

import os, re, sys, tarfile
import pandas as pd
import lxml.etree as ET

from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed, wait, FIRST_COMPLETED

# =============================================================================
# CONFIGURATION
# =============================================================================

MAX_SUCCESS_LIMIT = 100_000   # stop after X saved articles (overall limit)
MAX_WORKERS       = 7         # use 7 of 8 cores
MAX_QUEUE_SIZE    = 200       # RAM protection – max. gleichzeitig laufende Futures

TESTRUN     = False           # True = max 10 saved articles (quick check)
SAVE_PICKLE = True            # True = save raw pickle (full run only!)
SAVE_CSV    = True            # True = save CSV

SAMPLE_RATE = 190              # save every 190th SUCCESSFUL article (≈0.53%)

# Paths
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "big_data").exists() else _cwd.parent
BIG_DATA     = PROJECT_ROOT / "big_data"
DATA_DIR     = PROJECT_ROOT / "data"
PLOTS_DIR    = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

ARCHIVE_DIR = BIG_DATA / "PubMed"          # Ordner mit allen .tar.gz
ARCHIVES = {
    "baseline": BIG_DATA / "oa_noncomm_xml.PMC003xxxxxx.baseline.2026-01-23.tar.gz",
    "incr":     BIG_DATA / "oa_noncomm_xml.incr.2026-03-04.tar.gz",
}

RUN_TYPE = "baseline"   # nur für Fallback, falls ARCHIVE_DIR nicht existiert
FILEPATH = ARCHIVES[RUN_TYPE]

DTD_DIR = PROJECT_ROOT / "big_data" / "JATS-Archiving-1-3-MathML3-DTD"

# =============================================================================
# VALIDATION
# =============================================================================

if not DTD_DIR.exists():
    raise FileNotFoundError(f"DTD directory not found: {DTD_DIR}")
if not FILEPATH.exists():
    raise FileNotFoundError(f"Archive not found: {FILEPATH}")

BASE_URL = DTD_DIR.as_uri()
if not BASE_URL.endswith("/"):
    BASE_URL += "/"

print(f"DTD URL  : {BASE_URL}")

# =============================================================================
# REGEX (precompiled for performance)
# =============================================================================

RE_BRACKETS_SQ = re.compile(r'\[[\s,;\d-]*\]')
RE_BRACKETS_RD = re.compile(r'\([\s,;\d-]*\)')
RE_WHITESPACE  = re.compile(r'\s{2,}')

# =============================================================================
# HELPER
# =============================================================================

def clean_text_formatting(text: str) -> str:
    """Remove citation brackets [1,2] (3-5) and normalize whitespace."""
    if not text:
        return ""
    text = RE_BRACKETS_SQ.sub('', text)
    text = RE_BRACKETS_RD.sub('', text)
    text = RE_WHITESPACE.sub(' ', text)
    return text.strip()

# =============================================================================
# PARSER (module-level so worker processes can reuse it)
# =============================================================================

_parser = ET.XMLParser(dtd_validation=True, load_dtd=True, recover=True, no_network=True)

# =============================================================================
# WORKER (unverändert)
# =============================================================================

SECTION_MAP = {
    'introduction': ['introduction', 'background', 'context', 'rationale'],
    'methods':      ['method', 'material', 'implementation',
                     'experimental', 'procedure', 'methodology'],
    'results':      ['result', 'finding', 'observation', 'data analysis'],
    'discussion':   ['discussion', 'interpretation', 'comment'],
    'conclusion':   ['conclusion', 'summary', 'concluding', 'closing', 'outlook'],
}
NOISE_TITLES = {
    'conflict', 'funding', 'acknowledgment',
    'contribution', 'ethics', 'data availability'
}

def _extract_journal(root) -> str:
    paths = [
        '//journal-title-group/journal-title/text()',
        '//journal-title/text()',
        '//journal-id[@journal-id-type="nlm-ta"]/text()',
        '//journal-id[@journal-id-type="publisher-id"]/text()',
    ]
    for path in paths:
        hits = root.xpath(path)
        if hits and hits[0].strip():
            return hits[0].strip()
    return ""

def process_xml_worker(xml_content: bytes, filename: str):
    """
    Parse one XML file and return structured article data.

    Returns
    -------
    (paper_dict, None)   on success
    (None, error_dict)   on failure
    """
    try:
        root = ET.fromstring(xml_content, parser=_parser, base_url=BASE_URL)

        # Strip citations and MathML globally
        ET.strip_elements(root, 'xref', with_tail=True)
        ET.strip_elements(root, '{http://www.w3.org/1998/Math/MathML}*', with_tail=True)

        # Filter: research-article + English only
        article_type = root.get('article-type')
        lang_list    = root.xpath('//article/@xml:lang | //article-meta//@xml:lang')
        lang         = lang_list[0] if lang_list else "en"

        if article_type != "research-article" or not str(lang).startswith('en'):
            return None, {"filename": filename,
                          "error": f"not research-article: {article_type}"}

        paper = {col: "" for col in [
            'filename', 'pmcid', 'doi', 'journal', 'year', 'authors',
            'title', 'keywords', 'abstract',
            'introduction', 'methods', 'results', 'discussion', 'conclusion',
            'tables', 'metadata_junk'
        ]}
        paper['filename'] = filename.split('/')[-1]

        # Metadata
        paper['pmcid']   = "".join(root.xpath('//article-id[@pub-id-type="pmc"]/text()'))
        paper['doi']     = "".join(root.xpath('//article-id[@pub-id-type="doi"]/text()'))
        paper['journal'] = _extract_journal(root)
        paper['year'] = "".join(root.xpath('//pub-date//year/text()')[:1])

        auth_list = [
            f'{"".join(a.xpath("./surname/text()"))}'
            f' {"".join(a.xpath("./given-names/text()"))}'.strip()
            for a in root.xpath('//contrib[@contrib-type="author"]//name')
        ]
        paper['authors'] = "; ".join(auth_list)

        # Title
        paper['title'] = clean_text_formatting(
            " ".join(root.xpath('//article-title//text()')))

        # Keywords
        kwd_xpath = (
            '//kwd//text() | '
            '//compound-kwd-part//text() | '
            '//nested-kwd//text()'
        )
        raw_kwds = root.xpath(kwd_xpath)
        paper['keywords'] = "; ".join(sorted(set(
            k.strip() for k in raw_kwds if k.strip() and len(k.strip()) > 1
        )))

        if not paper['keywords']:
            return None, {"filename": filename, "error": "no keywords"}

        # Abstract
        abstract_paras = root.xpath('//abstract[not(@abstract-type)]//p')
        if not abstract_paras:
            abstract_paras = root.xpath('//abstract//p')
        paper['abstract'] = clean_text_formatting(
            ' '.join([' '.join(p.itertext()) for p in abstract_paras]))

        # Tables in Markdown
        tables = []
        for table_wrap in root.xpath('//table-wrap'):
            try:
                rows_data = []
                rows = table_wrap.xpath('.//tr')
                for row in rows:
                    row_cells = []
                    for th in row.xpath('.//th'):
                        cell_text = ' '.join(th.xpath('.//text()')).strip()
                        row_cells.append(clean_text_formatting(cell_text))
                    for td in row.xpath('.//td'):
                        cell_text = ' '.join(td.xpath('.//text()')).strip()
                        row_cells.append(clean_text_formatting(cell_text))
                    if row_cells:
                        rows_data.append(row_cells)
                if rows_data:
                    markdown_lines = []
                    for i, row in enumerate(rows_data):
                        markdown_row = '| ' + ' | '.join(row) + ' |'
                        markdown_lines.append(markdown_row)
                        if i == 0:
                            separator = '|' + '|'.join([' --- ' for _ in row]) + '|'
                            markdown_lines.append(separator)
                    table_markdown = '\n'.join(markdown_lines)
                    caption = table_wrap.xpath('.//caption//text()')
                    if caption:
                        caption_text = clean_text_formatting(' '.join(caption))
                        table_markdown = f"**{caption_text}**\n\n{table_markdown}"
                    tables.append(table_markdown)
            except Exception:
                continue
        paper['tables'] = tables

        # Sections
        for sec in root.xpath('//body/sec'):
            sec_title = " ".join(sec.xpath('./title/text()')).lower()
            sec_text  = clean_text_formatting(
                " ".join([" ".join(p.itertext()) for p in sec.xpath('.//p')]))

            if any(noise in sec_title for noise in NOISE_TITLES):
                paper['metadata_junk'] += f"\n[{sec_title}]: {sec_text}"
                continue

            assigned = False
            for target, keywords in SECTION_MAP.items():
                if any(k in sec_title for k in keywords):
                    paper[target] = (paper[target] + " " + sec_text).strip()
                    assigned = True
            if not assigned:
                paper['metadata_junk'] += f"\n[unmapped: {sec_title}]: {sec_text}"

        # Quality check
        required = ['abstract', 'introduction', 'methods', 'results', 'discussion']
        for req in required:
            if not paper[req] or len(paper[req]) < 100:
                return None, {"filename": filename,
                              "error": f"section {req} missing/short"}

        return paper, None

    except Exception as e:
        return None, {"filename": filename, "error": str(e)}


# =============================================================================
# MAIN EXTRACTION (mit Queue, regelmäßigem Feedback und erfolgsbasiertem Sampling)
# =============================================================================

def run_extraction(archive_path: Path, max_articles: int = None):
    extracted_data = []
    all_errors = []
    step = 190
    position = 0               # Zähler für alle XML-Dateien
    next_check = step - 1       # nächster Prüfpunkt (0-basiert: 189, 379, …)

    with tarfile.open(archive_path, "r:gz") as tar:
        # Nur XML-Dateien berücksichtigen
        members = [m for m in tar.getmembers() 
                   if m.isfile() and m.name.lower().endswith(('.nxml', '.xml'))]
        total = len(members)
        print(f"XML-Dateien im Archiv: {total}")

        # Sequentielle Verarbeitung (kein ProcessPool)
        for idx, member in enumerate(members):
            if idx == next_check:
                # Prüfpunkt erreicht – jetzt müssen wir einen erfolgreichen Artikel finden
                found = False
                while idx < total and not found:
                    f = tar.extractfile(members[idx])
                    if f:
                        res, err = process_xml_worker(f.read(), members[idx].name)
                        if err:
                            all_errors.append(err)
                        if res:
                            extracted_data.append(res)
                            found = True
                            if max_articles and len(extracted_data) >= max_articles:
                                break
                    else:
                        all_errors.append({"filename": members[idx].name, "error": "extraction failed"})
                    idx += 1   # nächste Datei, falls nicht erfolgreich
                # Nächsten Prüfpunkt setzen
                next_check += step
                if found:
                    # Wir sind jetzt bei idx (dem erfolgreichen Artikel) – der nächste Prüfpunkt muss 
                    # von diesem idx aus weiterspringen? Nein, wir setzen einfach den nächsten 
                    # absoluten Prüfpunkt: next_check = (idx // step + 1) * step + (step - 1)
                    # Aber einfacher: next_check = ((idx // step) + 1) * step - 1
                    next_check = ((idx // step) + 1) * step - 1
                # Abbruchbedingung
                if max_articles and len(extracted_data) >= max_articles:
                    break


    # Statistik
    total_files = position
    print(f"\nMembers processed  : {position}")
    print(f"XML files parsed   : {len(futures_in_order)}")
    print(f"Articles saved     : {len(extracted_data)} (target: every {SAMPLE_RATE}th file)")
    print(f"Successful total   : {success_count}")
    print(f"Articles dropped   : {len(all_errors)}")
    if total_files > 0:
        print(f"Success rate (raw) : {100 * success_count / (success_count + len(all_errors)):.1f} %")
        print(f"Sampling rate (vs total files): {100 * len(extracted_data) / total_files:.1f} % (target {100/SAMPLE_RATE:.1f}%)")

    return pd.DataFrame(extracted_data), pd.DataFrame(all_errors)


# =============================================================================
# ENTRY POINT (mehrere Archive verarbeiten, speichereffizient)
# =============================================================================
if __name__ == "__main__":

    all_articles = []   # sammelt alle erfolgreichen Artikel als Dicts
    all_errors   = []   # sammelt alle Fehlermeldungen als Dicts

    if ARCHIVE_DIR.exists():
        tar_files = sorted(ARCHIVE_DIR.glob("*.tar.gz"))
        print(f"Gefundene Archive: {len(tar_files)}")

        for tar_path in tar_files:
            print(f"\n=== Verarbeite {tar_path.name} ===")
            df, df_err = run_extraction(
                archive_path=tar_path,
                max_articles=10 if TESTRUN else None
            )
            if not df.empty:
                all_articles.extend(df.to_dict('records'))
            if not df_err.empty:
                all_errors.extend(df_err.to_dict('records'))
    else:
        # Fallback für Einzelarchiv (wie bisher)
        print("ARCHIVE_DIR nicht gefunden, verarbeite Einzelarchiv.")
        df, df_err = run_extraction(
            archive_path=FILEPATH,
            max_articles=10 if TESTRUN else None
        )
        if not df.empty:
            all_articles.extend(df.to_dict('records'))
        if not df_err.empty:
            all_errors.extend(df_err.to_dict('records'))

    # Kombinierte Ergebnisse speichern
    if all_articles:
        final_df = pd.DataFrame(all_articles)
        print(f"\nKombinierter Datensatz: {len(final_df)} Artikel")

        if SAVE_PICKLE and not TESTRUN:
            out_pkl = BIG_DATA / "combined" / "raw_articles_all.pkl"
            out_pkl.parent.mkdir(parents=True, exist_ok=True)
            final_df.to_pickle(out_pkl)
            print(f"Kombiniert gespeichert: {out_pkl}")

        if SAVE_CSV and not TESTRUN:
            out_csv = BIG_DATA / "combined" / "raw_articles_all.csv"
            out_csv.parent.mkdir(parents=True, exist_ok=True)
            final_df.to_csv(out_csv, index=False)
            print(f"Kombiniert gespeichert: {out_csv}")
    else:
        print("Keine Artikel extrahiert.")

    if all_errors:
        final_err = pd.DataFrame(all_errors)
        err_path = DATA_DIR / "extraction_errors_all.csv"
        err_path.parent.mkdir(parents=True, exist_ok=True)
        final_err.to_csv(err_path, index=False)
        print(f"Kombiniertes Fehler-Log: {len(final_err)} Einträge → {err_path}")